# LeIsaac VLA Play

来源：<https://lightwheelai.github.io/leisaac/resources/available_policy/>

**前提条件**（运行任何 cell 之前）：
1. 在 `isaaclab-experience/` 目录下启动 jupyter
2. 启动前已激活 conda 环境：`conda activate isaaclab`

> 各演示之间相互独立，建议一次只跑一组（推理 server + 客户端），跑完用对应章节的 stop cell 释放显存。


## 0) 预检查

In [ ]:
!cd LeIsaac && test -f scripts/evaluation/policy_inference.py && test -f assets/robots/so101_follower.usd && test -f assets/scenes/kitchen_with_orange/scene.usd && python scripts/evaluation/policy_inference.py --help | head -n 20

## 1) ⭐ GR00T N1.5 + LightwheelAI fine-tune（LeIsaac SO-101 PickOrange）

NVIDIA Isaac-GR00T N1.5（3B 参数，DiT diffusion action head）+ LightwheelAI 在仿真录制数据上 fine-tune 的 `leisaac-pick-orange-v0` checkpoint。
**实测成功率 1/1**：机器人能从初始位姿移动到 orange 上方、合爪抓取、移送到盘子并松开。

依赖（独立 conda env `gr00t-n15`，源码 `~/work/Isaac-GR00T-N1.5`）已在本机就绪——见 `MEMORY.md` 中的安装记录。


### 1.1) 一键下载 fine-tuned ckpt（~7.4 GB，已有则跳过）

In [ ]:
!bash scripts/download_gr00t_n15_ckpt.sh


### 1.2) 一键启动 GR00T N1.5 推理服务（ZMQ :5555）

冷启动需加载 Eagle backbone + DiT head，约 20–30s；已启动则 idempotent skip。


In [ ]:
!bash scripts/start_gr00t_n15_server.sh


### 1.3) 运行 SO-101 PickOrange 仿真推理

Isaac Sim 会弹窗显示 SO-101 单臂；客户端连 `:5555` 调 GR00T N1.5 推理。`-u` 是为了让 Episode/success 日志实时刷出来（Isaac Sim 退出会跳过 stdout flush）。


In [ ]:
!cd LeIsaac && PYTHONUNBUFFERED=1 python -u scripts/evaluation/policy_inference.py --task=LeIsaac-SO101-PickOrange-v0 --eval_rounds=1 --episode_length_s=120 --policy_type=gr00tn1.5 --policy_host=127.0.0.1 --policy_port=5555 --policy_timeout_ms=15000 --policy_language_instruction='Pick up the orange and place it on the plate' --policy_action_horizon=16 --device=cuda --enable_cameras


### 1.4) 停止 GR00T N1.5 推理服务（释放显存）

In [ ]:
!bash scripts/stop_gr00t_n15_server.sh


## 2) 服务端一键后台启动（推荐）

直接调用：`./server/start_server.sh [--gr00t-only|--lerobot-only]` / `./server/status_server.sh` / `./server/stop_server.sh`

In [ ]:
!GR00T_SIM_WRAPPER=1 bash server/start_server.sh


In [ ]:
!bash server/status_server.sh

In [ ]:
!bash server/stop_server.sh

## 3) GR00T N1.6 推理（GR1 双臂人形）

**模型适配场景**：`robocasa-gr1-tabletop`（24 个 PnP 任务，base 模型自带 GR1 head，可直接 zero-shot）

**不适配场景**：LeIsaac SO-101 系列 —— base 模型没有 SO-101 head，必须先 finetune。

## 3.1) 检查并启动 GR00T 推理服务


In [ ]:
!bash scripts/check_start_gr00t.sh


## 3.2) 实时预览推理（GR1 双臂 tabletop）

跑通后 `totem` 自动全屏播放生成的 mp4。任务可用 `ENV_NAME=...` 覆盖。


In [ ]:
!bash scripts/preview_gr00t_inference.sh


### ⚠ 注意：GR00T base 不能直接推理 LeIsaac SO-101

`GR00T-N1.6-3B` 内置的 EmbodimentTag 是 `GR1 / UNITREE_G1 / ROBOCASA_PANDA_OMRON / LIBERO_PANDA / OXE_GOOGLE / OXE_WIDOWX / OXE_DROID / BEHAVIOR_R1_PRO`，**没有 SO-101**。

要在 LeIsaac SO-101 任务上用 GR00T 推理，必须：
1. 在 SO-101 上采集（或获取）demonstration 数据，转成 LeRobot V2 schema；
2. 按 [`getting_started/finetune_new_embodiment.md`](../Isaac-GR00T/getting_started/finetune_new_embodiment.md) 用 `NEW_EMBODIMENT` tag finetune；
3. 用 finetuned checkpoint 重启 server（**去掉** `--use-sim-policy-wrapper`，那是给 robocasa 框架用的）；
4. 再回来跑 LeIsaac SO-101 推理。

本 cell **不可执行**。要看 GR00T 真跑机器人的效果，请回到 §2.2。

## 4) LeRobot SmolVLA 推理（LeIsaac SO-101 单臂）

**模型适配场景**：LeIsaac-SO101-* 系列任务（如 PickOrange、LiftCube、CleanToyTable…）。

⚠ `lerobot/smolvla_base` 是 HuggingFace LeRobot 团队发布的通用单臂 manipulation VLA，base ckpt **直接 zero-shot 推理不能让机器人完成 PickOrange**（实测：action chunk 持续输出但都在初始姿态附近 ±5° 微动，模型未在该任务上 fine-tune）。

本章节用于演示「客户端 → LeRobot v0.4 server → SmolVLA」整条推理链路打通；要看真实抓取效果请用 §1 GR00T N1.5 + LightwheelAI ckpt。


### 4.1) 启动 LeRobot 推理服务

端口 `:8080`。已起则 idempotent skip。


In [ ]:
!bash server/start_server.sh --lerobot-only


### 4.2) 安装 LeIsaac LeRobot client 依赖（首次执行后可跳过）


In [ ]:
!cd LeIsaac && pip install -e "source/leisaac[lerobot-async]"


### 4.3) 运行 SO-101 PickOrange 仿真推理

Isaac Sim 会弹窗显示 SO-101 单臂；客户端连 `:8080` 调 SmolVLA 推理。


In [ ]:
!cd LeIsaac && python scripts/evaluation/policy_inference.py --task=LeIsaac-SO101-PickOrange-v0 --eval_rounds=1 --policy_type=lerobot-smolvla --policy_host=127.0.0.1 --policy_port=8080 --policy_timeout_ms=15000 --policy_language_instruction='Pick the orange to the plate' --policy_checkpoint_path=lerobot/smolvla_base --policy_action_horizon=16 --device=cuda --enable_cameras


### 4.4) 停止 LeRobot 推理服务（释放显存）

不影响 GR00T `:5555`。


In [ ]:
!bash server/stop_server.sh --lerobot-only
